<a href="https://colab.research.google.com/github/natdebandi/hate_speech_ar/blob/main/4_analysis_hate_ar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Analisis del Dataset completo marzo 2020 - abril 2021 de X

**Natalia Dedandi**

Se aplica el clasificador seleccionado a todo el conjunto de datos recoelctado para analizar las características del odio en Argentina durante la pandemia

In [ ]:
%pip install datasets seaborn


In [ ]:
%pip install pandas
%pip install seaborn

Recupero el conjutno completo de datos: https://huggingface.co/datasets/piuba-bigdata/articles_and_comments



In [3]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv
from datasets import load_dataset
import pandas as pd

load_dotenv() 
token = os.getenv("HF_TOKEN")


dataset = load_dataset("piubamas/articles_and_comments", token=token)

c:\Users\Usuario\Documents\CIAI\Xenofobia_ar\xenofobia_ar\ciai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
#from datasets import load_dataset
#import pandas as pd

#dataset = load_dataset("piubamas/articles_and_comments")


In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['tweet_id', 'text', 'title', 'url', 'user', 'body', 'created_at', 'comments'],
        num_rows: 537201
    })
})

In [3]:
ds1 = dataset["train"]

Ejemplo de una fila

In [4]:
ds1[122]

{'tweet_id': '1376979520729849858',
 'text': 'Una fuerte apuesta a la rueda de la agroindustrialización argentina https://t.co/1o6ThCdtxw https://t.co/TLfGhLBjHu',
 'title': 'Una fuerte apuesta a la rueda de la agroindustrialización argentina',
 'url': 'https://www.lanacion.com.ar/economia/campo/una-fuerte-apuesta-a-la-rueda-de-la-agroindustrializacion-argentina-nid30032021/?utm_term=Autofeed&utm_medium=Echobox&utm_source=Twitter#Echobox=1617130198',
 'user': 'LANACION',
 'body': 'Desde el primer día, YPF Agro se propuso proveer al campo argentino de los insumos y las energías necesarias para producir más y mejor, en un contexto donde la demanda de alimentos para la población mundial sigue creciendo.\n\nEl negocio de agro de YPF cerró 2020 con ventas mayores a 1.428 millones de dólares y más de 1,5 millones de toneladas de grano canjeadas, y apuesta a incrementar estos resultados en 2021.\n\nEn ese sentido, lanzó su nueva campaña de comunicación orientada a reforzar la idea de circular

En este conjunto de datos tengo las etiquetas de la clasificación realizada con el modelo BETO:

https://huggingface.co/piuba-bigdata/beto-contextualized-hate-speech

Puede usarse para comparar pero en principio nos interesa recuperar los comentarios completos para aplicarle el clasificador seleccionado

In [5]:
from datetime import datetime, timedelta
from tqdm.auto import tqdm
from collections import Counter

In [7]:
tweets_arg = []

for noticia in tqdm(dataset['train']):

    date_noticia = noticia["created_at"]
    # Convertimos la fecha de la noticia a un objeto de python
    datenoticia = datetime.strptime(date_noticia, "%Y-%m-%dT%H:%M:%S%fZ")
    i=0
    for comment in noticia["comments"]:
        date_comment = comment["created_at"]
        # Convertimos la fecha del comentario a un objeto de python
        datecomment = datetime.strptime(date_comment, "%Y-%m-%dT%H:%M:%S%fZ")
        # anexa comentarios de diarios argentinos
        tweets_arg.append({"id":i,
            "tweet_id_noticia": noticia["tweet_id"],
            "title": noticia["title"],
            "resumen": noticia["text"],
            "medio": noticia["user"],
            "date_tweet": datecomment,
            "text": comment["text"],
            "user_id": comment["user_id"],
            "CALLS":comment["prediction"]["CALLS"],
            "WOMEN":comment["prediction"]["WOMEN"],
            "LGBTI":comment["prediction"]["LGBTI"],
            "RACISM":comment["prediction"]["RACISM"],
            "CLASS":comment["prediction"]["CLASS"],
            "POLITICS":comment["prediction"]["POLITICS"],
            "DISABLED":comment["prediction"]["DISABLED"],
            "APPEARANCE":comment["prediction"]["APPEARANCE"],
            "CRIMINAL":comment["prediction"]["CRIMINAL"]
            })
        i=i+1

100%|██████████| 537201/537201 [08:56<00:00, 1001.43it/s]


In [8]:
tweets_arg[11]

{'id': 3,
 'tweet_id_noticia': '1376940852011020288',
 'title': 'La desgarradora confesión de Hugo Maradona: “Lo sueño a Diego hablándome”',
 'resumen': 'La desgarradora confesión de Hugo Maradona: “Lo sueño a Diego hablándome” https://t.co/KKpQIYx9gD https://t.co/GN2kt809cl',
 'medio': 'LANACION',
 'date_tweet': datetime.datetime(2021, 3, 30, 17, 49, 1, 500000),
 'text': '@LANACION Re largo el sueño',
 'user_id': '395054756',
 'CALLS': 0,
 'WOMEN': 0,
 'LGBTI': 0,
 'RACISM': 0,
 'CLASS': 0,
 'POLITICS': 0,
 'DISABLED': 0,
 'APPEARANCE': 0,
 'CRIMINAL': 0}

In [9]:
# prompt: create a dataframe from tweets_arg

df_tweets_arg = pd.DataFrame(tweets_arg)


In [10]:
df_tweets_arg['HATEFUL']= df_tweets_arg[['CALLS','WOMEN','LGBTI','RACISM','CLASS','POLITICS','DISABLED','APPEARANCE','CRIMINAL']].max(axis=1)

In [11]:
df_tweets_arg.head()

,id,tweet_id_noticia,title,resumen,medio,date_tweet,text,user_id,CALLS,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,HATEFUL
0,0,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:03:00.900,@clarincom A mi me preocupa el trabajo.. La ev...,1532596098,0,0,0,0,0,0,0,0,0,0
1,1,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:05:04.500,@clarincom Lo que preocupa. https://t.co/Vmf9V...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
2,2,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:06:03.100,@clarincom Lo que les preocupa. https://t.co/P...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
3,3,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:11:02.300,@clarincom Le recomendaríamos al presidente de...,582286194,0,0,0,0,0,0,0,0,0,0
4,4,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:26:00.600,@clarincom Para salvar al correo de la quiebra...,951552761988034560,0,0,0,0,0,0,0,0,0,0


coloco el mismo orden de las etiquetas que usa el clasificador

In [12]:
# prompt: change the order of columns of df_tweets_arg and put HATEFUL before CALLS


df_tweets_arg = df_tweets_arg[['id', 'tweet_id_noticia','title','resumen','medio', 'date_tweet', 'text', 'user_id', 'HATEFUL', 'CALLS', 'WOMEN',
       'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE',
       'CRIMINAL']]
df_tweets_arg.head()

,id,tweet_id_noticia,title,resumen,medio,date_tweet,text,user_id,HATEFUL,CALLS,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL
0,0,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:03:00.900,@clarincom A mi me preocupa el trabajo.. La ev...,1532596098,0,0,0,0,0,0,0,0,0,0
1,1,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:05:04.500,@clarincom Lo que preocupa. https://t.co/Vmf9V...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
2,2,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:06:03.100,@clarincom Lo que les preocupa. https://t.co/P...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
3,3,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:11:02.300,@clarincom Le recomendaríamos al presidente de...,582286194,0,0,0,0,0,0,0,0,0,0
4,4,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:26:00.600,@clarincom Para salvar al correo de la quiebra...,951552761988034560,0,0,0,0,0,0,0,0,0,0


In [13]:
# prompt: save df_tweets_arg to csv file

df_tweets_arg.to_csv('data/tweets_con_noticia.csv', index=False)


In [14]:
# prompt: len of df_tweets_arg

len(df_tweets_arg)


8569648